# Практикум 17. Стресс-тест на морфологии

Сопровождает интерактивную лабораторию №17 («Стресс-тестирование NLP на агглютинативных языках»). Ноутбук самодостаточен: первая ячейка создаёт файл `nlp_common.py` с учебным корпусом русского языка (шаблонные предложения с морфологической разметкой) и вспомогательными функциями. Запускайте ячейки сверху вниз. Для реальной задачи замените учебный корпус своим.

In [ ]:
%%writefile nlp_common.py
"""Учебный модуль: генераторы корпусов для русского языка."""
import random, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110; plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .25
LANG = "ru"
NOUN = {  # лемма: (им., вин., предл.)
 "мама": ("мама","маму","маме"), "девочка": ("девочка","девочку","девочке"), "студентка": ("студентка","студентку","студентке"),
 "учительница": ("учительница","учительницу","учительнице"), "книга": ("книга","книгу","книге"), "газета": ("газета","газету","газете"),
 "песня": ("песня","песню","песне"), "картина": ("картина","картину","картине"), "школа": ("школа","школу","школе"),
 "библиотека": ("библиотека","библиотеку","библиотеке"), "комната": ("комната","комнату","комнате"), "улица": ("улица","улицу","улице")}
SUBJ = ["мама","девочка","студентка","учительница"]; OBJ = ["книга","газета","песня","картина"]; PLACE = ["школа","библиотека","комната","улица"]
ADJ_NOM = {"умный":"умная","добрый":"добрая","молодой":"молодая","весёлый":"весёлая"}
ADJ_ACC = {"новый":"новую","интересный":"интересную","красивый":"красивую","старый":"старую"}
VERB = {"читать":"читает","писать":"пишет","любить":"любит","видеть":"видит"}

def make_sentences(n=500, seed=0):
    """Предложения с морфологической разметкой: tokens, tags (UPOS), lemmas."""
    r = random.Random(seed); out = []
    for _ in range(n):
        t, g, l = [], [], []
        if r.random() < .5:
            a = r.choice(list(ADJ_NOM)); t.append(ADJ_NOM[a]); g.append("ADJ"); l.append(a)
        s = r.choice(SUBJ); t.append(NOUN[s][0]); g.append("NOUN"); l.append(s)
        v = r.choice(list(VERB)); t.append(VERB[v]); g.append("VERB"); l.append(v)
        if r.random() < .5:
            a = r.choice(list(ADJ_ACC)); t.append(ADJ_ACC[a]); g.append("ADJ"); l.append(a)
        o = r.choice(OBJ); t.append(NOUN[o][1]); g.append("NOUN"); l.append(o)
        if r.random() < .4:
            p = r.choice(PLACE); t += ["в", NOUN[p][2]]; g += ["ADP", "NOUN"]; l += ["в", p]
        out.append(dict(tokens=t, tags=g, lemmas=l))
    return out

def all_forms():
    """Все формы существительных с эталонной леммой: [(форма, лемма)]."""
    return sorted({(f, k) for k, v in NOUN.items() for f in v})

_A = ["кошка","собака","лошадь","корова"]; _F = ["рыбу","мясо","хлеб","сыр"]; _D = ["молоко","воду","чай","сок"]; _V = ["машина","автобус","поезд","трамвай"]
def make_topic_sentences(n=2000, seed=0):
    r = random.Random(seed); fr = [lambda: [r.choice(_A),"ест",r.choice(_F)], lambda: [r.choice(_A),"пьёт",r.choice(_D)], lambda: [r.choice(_A),"спит","дома"],
        lambda: [r.choice(_V),"едет","по","дороге"], lambda: [r.choice(_V),"стоит","у","остановки"]]
    return [r.choice(fr)() for _ in range(n)]
CATS = {w: c for c, ws in enumerate([_A, _F + _D, _V]) for w in ws}

def make_reviews(n=400, seed=0):
    r = random.Random(seed); N = ["фильм","товар","сервис","ужин","ремонт"]
    P = ["отличный","прекрасный","хороший","замечательный"]; B = ["ужасный","плохой","скучный","отвратительный"]; rows = []
    for _ in range(n):
        pos = r.random() < .5; adj = r.choice(P if pos else B); neg = r.random() < .3
        pat = r.choice(["{n} {a}","{n} очень {a}"]) if not neg else "{n} не {a}"
        rows.append((pat.format(n=r.choice(N), a=adj), int(pos != neg)))
    return pd.DataFrame(rows, columns=["text", "label"])

_PM = ["Иван","Ахмад","Фарход","Дмитрий","Алишер"]; _PF = ["Мария","Дильноза","Анна","Мадина"]
_SM = ["Петров","Каримов","Ахмедов","Иванов"]; _SF = [s + "а" for s in _SM]
_C = {"Ташкент":("Ташкент","Ташкенте"),"Самарканд":("Самарканд","Самарканде"),"Бухара":("Бухару","Бухаре"),"Москва":("Москву","Москве"),"Казань":("Казань","Казани")}
def make_ner(n=500, seed=0):
    """[(tokens, BIO-теги)]"""
    r = random.Random(seed); out = []
    for _ in range(n):
        f = r.random() < .5; c = r.choice(list(_C)); nm = r.choice(_PF if f else _PM); sn = r.choice(_SF if f else _SM)
        if r.random() < .5: t = [nm, sn, "приехала" if f else "приехал", "в", _C[c][0]]
        else: t = [nm, sn, "живёт", "в", _C[c][1]]
        out.append((t, ["B-PER","I-PER","O","O","B-LOC"]))
    return out

_PL_G = ["вокзала","аэропорта","центра","университета","рынка"]; _PL_A = ["вокзал","аэропорт","центр","университет","рынок"]
def make_intents(n=400, seed=0):
    r = random.Random(seed); rows = []
    T = {"book": ["хочу такси из {g} в {a}","закажите такси до {a}","нужно такси из {g}","вызовите такси в {a}","мне нужно такси","хочу поехать в {a}"],
         "cancel": ["отмените заказ","отмена заказа","хочу отменить такси"], "greet": ["привет","здравствуйте","добрый день"], "thanks": ["спасибо","благодарю вас","большое спасибо"]}
    for _ in range(n):
        k = r.choice(list(T)); rows.append((r.choice(T[k]).format(g=r.choice(_PL_G), a=r.choice(_PL_A)), k))
    return pd.DataFrame(rows, columns=["text", "intent"])

def make_parallel(n=200, seed=0):
    r = random.Random(seed); S = {"мама":"mother","папа":"father","девочка":"girl","студент":"student"}; V = {"читает":"reads","пишет":"writes","видит":"sees"}
    O = {"книгу":"book","газету":"newspaper","письмо":"letter","картину":"picture"}; out = []
    for _ in range(n):
        s, v, o = r.choice(list(S)), r.choice(list(V)), r.choice(list(O)); out.append(([s, v, o], [S[s], V[v], O[o]]))
    return out

ARTICLE = ["В Ташкенте открылась новая выставка современной живописи.","Выставка объединила работы тридцати художников из разных стран.",
 "Организаторы отметили, что выставка современной живописи привлекла много посетителей.","Художники представили картины, посвящённые городу и его жителям.",
 "Посетители особенно выделили работы молодых художников.","Выставка продлится до конца месяца, вход для студентов бесплатный.","Погода в городе в эти дни остаётся тёплой и солнечной."]
PASSAGES = ["Ташкент является столицей Узбекистана и крупнейшим городом страны с населением более двух миллионов человек.","Метро Ташкента открыто в 1977 году и стало первым метрополитеном в Средней Азии.",
 "Река Сырдарья протекает по территории Узбекистана, Казахстана и Таджикистана.","Самарканд известен древними мечетями и площадью Регистан.","Узбекская кухня славится пловом, который готовят в большом казане."]
QUESTIONS = [("Когда появился подземный транспорт в городе Ташкенте?", 1, "в 1977 году"), ("Какое блюдо считается визитной карточкой узбекской кухни?", 4, "плов"), ("Какая водная артерия пересекает несколько стран региона?", 2, "Сырдарья")]
AMBIG = [("Он взял ключ.", ["ключ от замка","родник","гаечный ключ"]), ("Я увидел человека с телескопом.", ["я смотрел в телескоп","у человека был телескоп"]), ("Эти стали прочнее.", ["сорта стали","сделались"])]

In [ ]:
from nlp_common import *

## 1. Рост словаря и незнакомые слова

In [ ]:
data = make_sentences(20000, seed=7); test = data[-3000:]; tr = data[:-3000]
tf = {w for s in test for w in s["tokens"]}; tl = {l for s in test for l in s["lemmas"]}
rows = []
for n in (100, 300, 1000, 3000, 10000, 17000):
    sub = tr[:n]; sw = {w for s in sub for w in s["tokens"]}; sl = {l for s in sub for l in s["lemmas"]}
    tt = [w for s in test for w in s["tokens"]]; ll = [l for s in test for l in s["lemmas"]]
    rows.append((n, len(sw), np.mean([w not in sw for w in tt]), np.mean([l not in sl for l in ll])))
df = pd.DataFrame(rows, columns=['предложений', 'разных форм', 'доля незнакомых форм', 'доля незнакомых лемм']); print(df.round(3))
df.plot(x='предложений', y=['доля незнакомых форм', 'доля незнакомых лемм'], logx=True, marker="o"); plt.show()

## 1б. Симуляция: сколько форм у слова в разных языках

In [ ]:
rng = np.random.default_rng(0); NL = 300
def oov_curve(F, sizes=(100, 300, 1000, 3000, 10000, 30000), test_n=5000):
    lw = 1 / np.arange(1, NL + 1); lw /= lw.sum(); fw = 1 / np.arange(1, F + 1) ** .6; fw /= fw.sum()
    draw = lambda n: (rng.choice(NL, n, p=lw), rng.choice(F, n, p=fw)); tl, tf = draw(test_n); out = []
    for n in sizes:
        l, f = draw(n); seen = set(zip(l.tolist(), f.tolist())); out.append(np.mean([(a, b) not in seen for a, b in zip(tl.tolist(), tf.tolist())]))
    return out
sizes = (100, 300, 1000, 3000, 10000, 30000)
for F, name in ((2, 'английский'), (12, 'русский'), (72, 'узбекский')): plt.plot(sizes, oov_curve(F), marker="o", label=f"{name}, F={F}")
plt.xscale("log"); plt.xlabel('слов в обучении'); plt.ylabel('доля незнакомых форм'); plt.legend(); plt.show()

## 2. Токенизация подсловами: токенов на слово

In [ ]:
from collections import Counter
def learn_bpe(words, n_merges):
    vocab = Counter(tuple(w) + ("_",) for w in words); merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for w, f in vocab.items():
            for a, b in zip(w, w[1:]): pairs[(a, b)] += f
        if not pairs: break
        best = max(pairs, key=lambda p: (pairs[p], p)); merges.append(best); new = Counter()
        for w, f in vocab.items():
            o, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and (w[i], w[i+1]) == best: o.append(w[i] + w[i+1]); i += 2
                else: o.append(w[i]); i += 1
            new[tuple(o)] += f
        vocab = new
    return merges
def bpe(word, merges):
    s = list(word) + ["_"]
    for a, b in merges:
        o, i = [], 0
        while i < len(s):
            if i < len(s) - 1 and (s[i], s[i+1]) == (a, b): o.append(a + b); i += 2
            else: o.append(s[i]); i += 1
        s = o
    return s
train_w = [w for s in tr[:3000] for w in s["tokens"]]; test_w = [w for s in test for w in s["tokens"]]
for m in (0, 20, 60, 150): mg = learn_bpe(train_w, m); print('слияний:', m, "|", 'токенов на слово:', round(np.mean([len(bpe(w, mg)) for w in test_w]), 2))

## 3. Стеммер на всех формах

In [ ]:
forms = all_forms()
if LANG == "ru":
    from nltk.stem.snowball import SnowballStemmer
    st = SnowballStemmer("russian"); by = {}
    for f, l in forms: by.setdefault(l, set()).add(st.stem(f))
    print('Лемм, чьи формы получили разные основы:', sum(len(v) > 1 for v in by.values()), "/", len(by))
else:
    CASES = ["ning", "dagi", "gacha", "ni", "ga", "da", "dan"]; POSS = ["imiz", "ingiz", "im", "ing", "i"]
    def strip(w):
        for L_ in (CASES, POSS, ["lar"]):
            c = next((s for s in sorted(L_, key=len, reverse=True) if w.endswith(s) and len(w) - len(s) >= 2), None)
            if c: w = w[:-len(c)]
        return w
    acc = {}
    for f, l in forms: acc.setdefault(l, []).append(strip(f) == l)
    print('Корни с ошибками стеммера:', {l: round(float(np.mean(v)), 2) for l, v in acc.items() if np.mean(v) < 1})

## Задания

1. Сравните кривые незнакомых форм для разных размеров корпуса. На каком размере доля падает ниже 5%?
2. Постройте токенизатор: слово → основа + суффиксы по правилам и сравните с BPE.
3. Придумайте три «стресс-теста» для своей задачи (длинные цепочки, редкие формы, чередования).